[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skous2/is455-w2027/blob/main/module_04/Worksheet_4A_Getting_and_Cleaning_Data.ipynb)

# Worksheet 4A · Getting and cleaning data

**Optional and not graded.** Work through it before Tuesday's class (Feb 2). Tuesday's walk-through covers sections 1–4 on getting data from an API, and Thursday's class covers sections 5–9 on cleaning it. Expect about 35–45 minutes.

**Before you start,** choose **File → Save a copy in Drive** so your work is saved. Try each problem yourself first. When you get stuck, paste the tutor prompt below into your AI assistant so it gives you hints instead of the answer. Write answers in your own words.

Each section starts with a short worked example from a made-up smoothie shop, then gives you the same kind of problem for Canyon Fitness. Each section also links the reading that covers it, and you can read each one when you reach its section.

## When you get stuck

Copy the whole prompt below into your AI assistant and fill in the two parts in brackets. It tells the AI to explain the idea and give one hint at a time instead of writing the code for you. After you solve the problem, it gives you one new practice problem to try on your own. If your AI tool can edit this notebook directly, tell it not to, so you type the code yourself.

```text
I'm a student in IS 455, a machine learning class, learning Python and pandas. Do not write the solution for me.
Here is the problem and my solution so far. Explain the idea I'm missing in plain language, give me one hint at a time,
and wait for me to try again after each hint.
When I have solved it, give me ONE new practice problem that tests the same idea with different data and a small twist.
Don't help with the practice problem unless I ask.

PROBLEM:
[paste the problem text]

MY CODE AND THE ERROR OR RESULT:
[paste your code and what happened]
```

## Where the data comes from

The clean Canyon Fitness file you've used so far had to come from somewhere. Canyon Fitness's member system has an **API**, a web address that returns data when your code requests it, and someone had to pull the records from it and clean them. This week you do both.

For this class, the API's responses are saved as files on the course GitHub, so everyone gets the same data and the service can't go down during class. Your code works exactly as it would with a live API, because you still request a web address and get data back in **JSON**, a text format made of nested lists and dictionaries.

## 1. Make one request

*Read [requests Quickstart, "Make a Request," "JSON Response Content" and "Response Status Codes"](https://requests.readthedocs.io/en/latest/user/quickstart/). McKinney 6.3, "Interacting with Web APIs", uses the same tools.*

**Worked example.** `requests.get` asks a web address for data. The response's **status code** says whether it worked. 200 means success and 404 means the address doesn't exist. `.json()` turns the JSON text into Python dictionaries and lists. `timeout=10` stops your code from waiting forever if the server doesn't answer.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/smoothie/orders_page_1.json"
response = requests.get(url, timeout=10)
print(response.status_code)      # 200 means the request worked

page = response.json()           # JSON text becomes a Python dictionary
print(page.keys())
print(page["results"][0])        # the first order on this page

**Your turn.**

Run the cell below to store the address of the Canyon Fitness API.

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/canyon/"
first_url = BASE_URL + "members_page_1.json"

**1a.** Request `first_url`. Print the status code, the keys of the response, `total_records`, `total_pages`, and the first member in `results`.

In [ ]:
# Your code here


**1b.** Request `BASE_URL + "members_page_99.json"`, a page that doesn't exist, and print its status code. Don't call `.json()` on it. What does the code mean, and why should you check the status code before calling `.json()`?

In [ ]:
# Your code here


*Your answer here.*

## 2. Get every page

*Read [requests Quickstart, "Passing Parameters In URLs" and "Timeouts"](https://requests.readthedocs.io/en/latest/user/quickstart/).*

**Worked example.** APIs usually return large results in pages. This one puts the address of the next page in each response's `"next"` field, and the last page has `"next": null`, which Python reads as `None`. A `while` loop keeps requesting pages until there isn't a next one. `raise_for_status()` stops with an error if a request fails, and `time.sleep` pauses between requests so you stay under the API's **rate limit**, the number of requests it allows in a given time. Many other APIs take the page number as a parameter instead, such as `requests.get(url, params={"page": 2})`.

In [ ]:
import time

orders = []
url = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/smoothie/orders_page_1.json"
while url is not None:
    response = requests.get(url, timeout=10)
    response.raise_for_status()          # stop here if the request failed
    page = response.json()
    orders.extend(page["results"])       # add this page's records to the list
    url = page["next"]                   # None on the last page, which ends the loop
    time.sleep(0.2)                      # pause so you don't overload the server

print(len(orders))

**Your turn.**

**2a.** Collect every member record from every page into a list called `records`. Print how many records you collected and compare it with `total_records` from page 1.

In [ ]:
# Your code here


**2b.** Why pause between requests? What could happen to a script that sends thousands of requests as fast as it can?

*Your answer here.*

## 3. Turn JSON into a DataFrame

*Read the [pandas documentation for `json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).*

**Worked example.** Each record is a dictionary, and some values are dictionaries themselves. `pd.json_normalize` turns a list of records into a DataFrame and flattens nested fields into columns with dots in their names, such as `customer.name`. `rename` then gives the columns simpler names.

In [ ]:
orders_df = pd.json_normalize(orders)
display(orders_df)

orders_df = orders_df.rename(columns={"customer.name": "customer", "customer.member": "member"})
print(list(orders_df.columns))

**Your turn.**

**3a.** Turn `records` into a DataFrame named `members` with `pd.json_normalize`. Rename these columns so they match the clean file you used in Modules 2 and 3, then print the shape and the column names.

| API name | New name |
| --- | --- |
| `joined` | `join_date` |
| `plan.name` | `plan` |
| `plan.monthly_fee` | `monthly_fee` |
| `visits.last_30_days` | `visits_last_30_days` |
| `visits.prior_30_days` | `visits_prior_30_days` |
| `visits.days_since_last` | `days_since_last_visit` |

In [ ]:
# Your code here


## 4. Keep API keys out of your notebook

*No reading for this section.*

**Worked example.** Most real APIs require an **API key**, a password-like string that identifies you and counts your requests. Never type a key into a notebook cell. Notebooks get shared and pushed to GitHub, and automated tools scan GitHub for leaked keys. In Colab, save the key in the **Secrets** panel, the key icon in the left sidebar, and read it in code with `from google.colab import userdata` and then `api_key = userdata.get("SMOOTHIE_API_KEY")`. The key never appears in the notebook itself.

**Your turn.**

**4a.** A teammate pastes `api_key = "sk_live_48f2..."` into a shared notebook and pushes it to GitHub. What could go wrong, and what should they do now?

*Your answer here.*

## 5. Duplicate records

*McKinney 7.2, "Data Transformation," covers this under "Removing Duplicates."*

**Worked example.** `duplicated()` marks each row that repeats an earlier row exactly, so `.sum()` counts them. Checking the ID column alone catches records that repeat an ID but differ somewhere else. `drop_duplicates()` keeps the first copy of each exact duplicate.

In [ ]:
sales = pd.DataFrame({"order_id": [1, 2, 2, 3], "drink": ["Mango", "Berry", "Berry", "Mango"]})
print(sales.duplicated().sum())               # rows that exactly repeat an earlier row
print(sales["order_id"].duplicated().sum())   # IDs that appear more than once
sales = sales.drop_duplicates()
print(len(sales))

**Your turn.**

**5a.** Count the exact duplicate rows in `members` and the repeated `member_id` values. Drop the exact duplicates, then confirm that every `member_id` is now unique and print the number of rows.

In [ ]:
# Your code here


## 6. Text that should be numbers or dates

*McKinney 7.4, "String Manipulation," covers `.str` functions, and McKinney 11.1 covers converting text to dates.*

**Worked example.** A dollar sign makes pandas read a price as text, so math on it fails. `.str.replace` removes the sign and `.astype(float)` turns the rest into a number. Dates written two different ways need `pd.to_datetime` with `format="mixed"`, which reads each value's format separately.

In [ ]:
prices = pd.DataFrame({"price": ["$4.75", "$5.25"], "date": ["2026-09-28", "09/30/2026"]})
prices["price"] = prices["price"].str.replace("$", "").astype(float)
prices["date"] = pd.to_datetime(prices["date"], format="mixed")
print(prices.dtypes)
prices

**Your turn.**

**6a.** Count how many `join_date` values contain a `/`, which marks the month/day/year format. Then convert `monthly_fee` to a number and convert `join_date`, `next_visit_date` and `cancel_date` to dates. Print the data types to check.

In [ ]:
# Your code here


## 7. Inconsistent categories and yes/no text

*McKinney 7.4 covers `.str.strip`, `.str.title` and `.str.upper`. McKinney 7.2 covers changing values under "Transforming Data with a Function or Mapping."*

**Worked example.** `value_counts` exposes spellings that should be the same category. `.str.strip()` removes spaces at the ends and `.str.title()` capitalizes the first letter of each word. Yes/no text becomes `True`/`False` by checking whether the uppercase value starts with "Y".

In [ ]:
drinks = pd.DataFrame({"drink": ["Mango", "mango ", "MANGO", "Berry"], "member": ["Y", "N", "Yes", "no"]})
print(drinks["drink"].value_counts())

drinks["drink"] = drinks["drink"].str.strip().str.title()
drinks["member"] = drinks["member"].str.upper().str.startswith("Y")
print(drinks["drink"].value_counts())
drinks

**Your turn.**

**7a.** Show `value_counts` for `home_gym`, then fix it so each gym has one spelling and show `value_counts` again. Then convert `autopay` to `True`/`False`.

In [ ]:
# Your code here


## 8. Placeholder codes and impossible values

*McKinney 7.2 covers this under "Replace Values" and "Detecting and Filtering Outliers." McKinney 7.1 covers missing data.*

**Worked example.** Some systems store a **placeholder code** such as -1 or 999 when a value is unknown. Left in place, a placeholder distorts every average, so it should become missing (`NaN`). An **impossible value**, such as an age of 240, can't be fixed by guessing, so it becomes missing too. A combination can also be impossible even when each value looks fine, such as a refund dated before its order. `.loc[condition, column] = ...` changes only the rows where the condition is `True`.

In [ ]:
import numpy as np

sales = pd.DataFrame({
    "age":         [34, -1, 27, 240],
    "order_date":  pd.to_datetime(["2026-09-28", "2026-09-28", "2026-09-29", "2026-09-30"]),
    "refund_date": pd.to_datetime([None, "2026-09-20", None, None]),
})
sales.loc[(sales["age"] < 0) | (sales["age"] > 110), "age"] = np.nan        # -1 means unknown; 240 is a typo
sales.loc[sales["refund_date"] < sales["order_date"], "refund_date"] = pd.NaT   # a refund can't come before the order
sales

**Your turn.**

**8a.** Count the members whose `age` is below 0 or above 110, then set those ages to missing.

In [ ]:
# Your code here


**8b.** A member can't visit after cancelling. Count the members whose `next_visit_date` is after their `cancel_date`, then set those `next_visit_date` values to missing. Why is it safer to blank the visit date than to delete the member?

In [ ]:
# Your code here


*Your answer here.*

## 9. Check your work

Run the cell below. It loads the clean file from Modules 2 and 3 and compares it with your cleaned data, column by column. If your cleaning matches, every column shows 0 differences except `age`, where the ages you set to missing differ on purpose. If another column shows differences, go back to the section that cleans it.

In [ ]:
clean = pd.read_csv("https://raw.githubusercontent.com/skous2/is455-w2027/main/data/canyon_fitness_members.csv", parse_dates=["join_date", "next_visit_date", "cancel_date"])
print(f"Your rows: {len(members)}   Clean file rows: {len(clean)}")

if len(members) == len(clean):
    mine = members.sort_values("member_id").reset_index(drop=True)
    for col in clean.columns:
        same = (mine[col] == clean[col]) | (mine[col].isna() & clean[col].isna())
        print(f"{col:30} {(~same).sum():>4} rows differ")

## Bring to class

Write down anything that confused you. Tuesday's walk-through covers sections 1–4, Thursday's class covers sections 5–9, and the answer key is posted after Tuesday's class.

*Your answer here.*